# Pretrained GPT-2: loading, probing, and fine-tuning

The previous notebook trained a small model from scratch. This one does something more
interesting: it loads **OpenAI's released GPT-2 weights into this from-scratch
implementation** and shows the two are the same model.

Then it fine-tunes that 124M model on Shakespeare — on a free T4, in a few minutes.

In [ ]:
# Colab setup. Skip the clone if you are running this locally from the repo.
import os, sys

if not os.path.exists("gpt2-from-scratch"):
    !git clone -q https://github.com/BharathBagadhi/gpt2-from-scratch.git
%cd gpt2-from-scratch
!pip install -q -e . 2>/dev/null

sys.path.insert(0, "src")

import torch
print("torch", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers

## 1. Load OpenAI's weights into our model

`transformers` is used here purely as a weight *downloader*. None of its layers run in the
forward pass — every tensor operation comes from `src/gpt2/model.py`.

The one wrinkle: the original GPT-2 was written in TensorFlow using `Conv1D`, whose weight is
stored transposed relative to `nn.Linear`. Four tensors per block need a `.t()` on the way in.
Everything else is a straight name-for-name copy, because the module names in `model.py` were
chosen to match the checkpoint.

In [ ]:
from gpt2.pretrained import load_pretrained
from gpt2.tokenizer import BPETokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_pretrained("gpt2").to(device).eval()
enc = BPETokenizer()

print(f"loaded GPT-2 small: {model.num_params():,} parameters")

## 2. Prove it is really GPT-2

Same tokens into both implementations; the logits must match to float tolerance. This is the
test that would fail on a wrong scale factor, a transposed matrix, a missing GELU
approximation, or an off-by-one in the positional embedding.

In [ ]:
from transformers import GPT2LMHeadModel

reference = GPT2LMHeadModel.from_pretrained("gpt2").to(device).eval()
idx = torch.randint(0, 50257, (2, 32), device=device)

with torch.no_grad():
    ours, _ = model(idx, targets=idx)
    theirs = reference(idx).logits

diff = (ours - theirs).abs().max().item()
print(f"max |ours - huggingface| = {diff:.3e}")
assert diff < 1e-3
print("identical to float tolerance")

## 3. What it knows

Greedy decoding (`temperature=0`) shows the model's single most confident continuation, which
is the clearest way to see what it has actually memorised.

In [ ]:
prompts = [
    "The capital of France is",
    "Water boils at a temperature of",
    "The theory of relativity was developed by",
    "In 1969, humans first landed on",
]

for p in prompts:
    idx = torch.tensor([enc.encode(p)], device=device)
    out = model.generate(idx, max_new_tokens=12, temperature=0.0)
    print(f"{p!r}\n  -> {enc.decode(out[0].tolist())}\n")

### Look inside the distribution

Rather than only sampling, inspect what the model believes. This is a more honest picture of
a 124M model than a cherry-picked generation.

In [ ]:
import torch.nn.functional as F

prompt = "The capital of France is"
idx = torch.tensor([enc.encode(prompt)], device=device)
with torch.no_grad():
    logits, _ = model(idx)

probs = F.softmax(logits[0, -1], dim=-1)
top = torch.topk(probs, 10)
print(f"{prompt!r} -> next token:\n")
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    bar = "#" * int(p * 60)
    print(f"  {enc.decode([i])!r:<14} {p:6.2%}  {bar}")

### Sampling knobs

`temperature`, `top_k` and `top_p` all shape the same distribution differently. Worth seeing
side by side.

In [ ]:
prompt = "The most important thing about machine learning is"
idx = torch.tensor([enc.encode(prompt)], device=device)

settings = [
    ("greedy",            dict(temperature=0.0)),
    ("temp 0.7, top-k 40", dict(temperature=0.7, top_k=40)),
    ("temp 1.0, top-p 0.9", dict(temperature=1.0, top_p=0.9)),
    ("temp 1.4 (chaotic)", dict(temperature=1.4, top_k=100)),
]

for name, kw in settings:
    torch.manual_seed(0)
    out = model.generate(idx, max_new_tokens=50, **kw)
    print(f"--- {name} ---\n{enc.decode(out[0].tolist())}\n")

## 4. Where GPT-2 124M actually lands

HellaSwag is the standard sanity benchmark: a context plus four candidate endings, scored by
average token log-likelihood. Random is 25%. GPT-2 124M gets about **29-30%** — barely above
chance.

That is the honest number, and reporting it matters more than hiding it. 124M parameters is
small; the interesting result is that it is above chance at all.

In [ ]:
from gpt2.evaluate import download_hellaswag, hellaswag_accuracy

path = download_hellaswag("data/hellaswag_val.jsonl")
result = hellaswag_accuracy(model, path, device=device, limit=300, enc=enc)

print(f"evaluated on {result['n']} examples")
print(f"  acc      : {result['acc']:.1%}")
print(f"  acc_norm : {result['acc_norm']:.1%}   (length-normalised, the quoted metric)")
print(f"  random   : 25.0%")

## 5. Fine-tune it on Shakespeare

Two changes from pretraining:

- **Learning rate drops ~20×** (6e-4 → 3e-5). The weights are already good; a pretraining-size
  step destroys them.
- **Dropout goes to 0.1.** 1 MB of text will otherwise be memorised outright.

`--block_size 256` crops the context window, which cuts memory and speeds this up a lot for a
fine-tune where long-range context is not the point.

In [ ]:
!python scripts/prepare_data.py --dataset tinyshakespeare

In [ ]:
!python scripts/train.py \
    --init_from gpt2 \
    --data_dir data/tinyshakespeare \
    --block_size 256 --batch_size 2 --grad_accum_steps 8 \
    --learning_rate 3e-5 --warmup_steps 50 --max_steps 400 \
    --dropout 0.1 --eval_interval 100 --eval_iters 20 \
    --out_dir out/finetune

## 6. Before and after

The same prompt through the base model and the fine-tuned one. The base model continues in
generic modern English; the fine-tuned one adopts the register, the speaker labels and the
line structure of a play — while keeping the grammar it learned from the web.

That is what fine-tuning does: it moves the *style*, not the *competence*.

In [ ]:
from gpt2.pretrained import load_checkpoint

tuned = load_checkpoint("out/finetune/ckpt_final.pt", device=device)
prompt = "ROMEO:"
idx = torch.tensor([enc.encode(prompt)], device=device)

for name, m in (("base GPT-2", model), ("fine-tuned", tuned)):
    torch.manual_seed(1337)
    out = m.generate(idx, max_new_tokens=180, temperature=0.8, top_k=50)
    print(f"{'=' * 70}\n{name}\n{'=' * 70}\n{enc.decode(out[0].tolist())}\n")

## 7. Throughput

Useful to know what your hardware actually delivers before planning a longer run.

In [ ]:
!python scripts/benchmark.py --preset gpt2 --batch_size 4 --block_size 512 --steps 10

## Where to go next

- **Multi-GPU**: `torchrun --standalone --nproc_per_node=N scripts/train.py --strategy ddp`
- **Real pretraining**: swap TinyShakespeare for FineWeb-Edu. Reproducing GPT-2 124M properly
  takes about 10B tokens and ~2 hours on 8×A100 (roughly $25-50 of rented GPU) — everything
  else in this repo is free.
- **Read the code**: `src/gpt2/model.py` is the whole architecture in about 300 lines.